In [ ]:
import pandas as pd

In [ ]:
import numpy as np
import obonet
import networkx as nx
import matplotlib.pyplot as plt
from skfp.fingerprints import RDKit2DDescriptorsFingerprint, MACCSFingerprint, ECFPFingerprint
from tqdm import tqdm
import torch
from sklearn.metrics import f1_score
from transformers import AutoTokenizer, AutoModel
import torch.nn as nn

In [ ]:
data_train = pd.read_parquet('/content/chebi_dataset_train.parquet')

In [ ]:
data_train

,mol_id,SMILES,class_0,class_1,class_2,class_3,class_4,class_5,class_6,class_7,...,class_490,class_491,class_492,class_493,class_494,class_495,class_496,class_497,class_498,class_499
0,mol_12500,CCCCC/C=C\CCCCCCCC(=O)O,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
1,mol_15962,Cc1cc2cc(O)cc(O)c2c(C)n1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
2,mol_42147,C[C@H](CCC[C@@H](C)C=O)[C@H]1CC[C@H]2[C@@H]3CC...,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
3,mol_43459,CCN=C1C=CC(=C(c2ccc(NCC)cc2)c2ccc(NCC)c(C)c2)C=C1,1,1,1,1,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
4,mol_12734,C[C@H](CCC(O)=N[C@@H](Cc1c[nH]c2ccccc12)C(=O)O...,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33663,mol_26342,C[C@H](OP(=O)(O)OC[C@@H](O)[C@@H](O)[C@@H](O)C...,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
33664,mol_5233,O=C(Nc1ccc(Cl)cc1)Nc1ccc(Cl)cc1,1,1,1,1,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
33665,mol_36641,Cc1cn([C@H]2C[C@H](OP(=O)(O)OC[C@H]3O[C@@H](n4...,1,1,1,1,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
33666,mol_9633,O=C([O-])C(O)Cc1c[nH]c2ccccc12,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
smiles_list = data_train['SMILES'].tolist()

In [ ]:
df_final = pd.read_csv('/content/final_molecule_data_with_chemberta.csv')

FileNotFoundError: [Errno 2] No such file or directory: '/content/final_molecule_data_with_chemberta.csv'

In [ ]:
target_cols = [col for col in data_train.columns if col.startswith('class_')]
X = df_final.drop(columns=['mol_id']) # Usuwamy ID, zostawiamy same cechy
y = data_train[target_cols]

print("Gotowe do treningu!")
print(f"X shape: {X.shape} (Cechy: RDKit2D + MACCS + ECFP4 + ChemBERTa)")
print(f"y shape: {y.shape} (Etykiety klas)")

Gotowe do treningu!
X shape: (33668, 2798) (Cechy: RDKit2D + MACCS + ECFP4 + ChemBERTa)
y shape: (33668, 500) (Etykiety klas)


In [ ]:
selected_cols = [col for col in X.columns if col.startswith(('ChemBERTa', 'MACCS'))]

# 2. Sortowanie numeryczne - bardzo ważne!
# Sortujemy najpierw po nazwie grupy, a potem po numerze po '_'
# Dzięki temu masz najpierw ChemBERTa_0...383, a potem MACCS_0...166 (lub odwrotnie)
selected_cols.sort(key=lambda x: (x.rsplit('_', 1)[0], int(x.rsplit('_', 1)[-1])))

# 3. Obcinamy DataFrame do wybranych cech
X_train_final = X[selected_cols].copy()

# 4. AKTUALIZACJA: Sprawdź nowy wymiar wejściowy
new_input_dim = X_train_final.shape[1]

print(f"Liczba cech po połączeniu (ChemBERTa + MACCS): {new_input_dim}")


Liczba cech po połączeniu (ChemBERTa + MACCS): 550


In [ ]:
class_names = X_train_final.columns

In [ ]:
def hierarchical_loss(outputs, hierarchy_indices, lambda_hier=0.1):
    """
    Karanie za sytuację, gdy P(dziecko) > P(rodzic)
    """
    if len(hierarchy_indices) == 0:
        return 0.0

    probs = torch.sigmoid(outputs)
    parents = hierarchy_indices[:, 0]
    children = hierarchy_indices[:, 1]

    # Obliczamy różnicę: Prob(Child) - Prob(Parent)
    # Jeśli różnica > 0, oznacza naruszenie hierarchii
    diff = probs[:, children] - probs[:, parents]

    # ReLu wyciąga tylko wartości dodatnie (naruszenia)
    violation_loss = torch.mean(torch.relu(diff))

    return lambda_hier * violation_loss

In [ ]:
LAMBDA_HIER = 0.5

In [ ]:
class MultiLabelNet(nn.Module):
    def __init__(self, input_dim=549, output_dim=500):
        super(MultiLabelNet, self).__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.LayerNorm(512), # Stabilizuje trening i radzi sobie z różnymi rozkładami cech
            nn.Dropout(0.3),

            nn.Linear(512, 512),
            nn.ReLU(),
            nn.LayerNorm(512),
            nn.Dropout(0.3),

            nn.Linear(512, output_dim) # Wyjście to 500 surowych logitów
        )

    def forward(self, x):
        return self.network(x)

In [ ]:
y_tensor = torch.tensor(y.values, dtype=torch.float32)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train_final.values, y.values, test_size=0.2, random_state=42
)

# Konwersja do tensorów (float32 jest wymagane przez PyTorch)
X_train_t = torch.tensor(X_train_final.values, dtype=torch.float32)
y_train_t = torch.tensor(y.values, dtype=torch.float32)



BATCH_SIZE = 32
full_train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(full_train_dataset, batch_size=BATCH_SIZE, shuffle=True)


val_loader = train_loader

def get_metrics(model, loader, device, criterion):
    model.eval()
    all_targets = []
    all_preds = []
    total_loss = 0
    criterion = nn.BCEWithLogitsLoss()

    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            total_loss += loss.item()

            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).int()

            all_targets.append(targets.cpu().int().numpy())
            all_preds.append(preds.cpu().int().numpy())

    all_targets = np.vstack(all_targets)
    all_preds = np.vstack(all_preds)

    # Obliczamy Macro-F1 (średnia po 500 klasach)
    macro_f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)
    avg_loss = total_loss / len(loader)

    return avg_loss, macro_f1

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MultiLabelNet(X_train_final.shape[1], y.shape[1]).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 3. Pętla treningowa z pełnymi logami
EPOCHS = 40
best_val_f1 = 0

print(f"{'Epoch':^7} | {'Train Loss':^10} | {'Val Loss':^10} | {'Train F1':^10} | {'Val F1':^10}")
print("-" * 60)

for epoch in range(EPOCHS):
    model.train()
    train_running_loss = 0.0

    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        base_loss = criterion(outputs, targets)

        base_loss.backward()
        optimizer.step()
        train_running_loss += base_loss.item()

    # Ewaluacja po epoce
    train_loss, train_f1 = get_metrics(model, train_loader, device, criterion)
    val_loss, val_f1 = get_metrics(model, val_loader, device, criterion)

    # LOGOWANIE
    print(f"{epoch+1:^7} | {train_loss:^8.4f} | {val_loss:^8.4f} | {train_f1:^8.4f} | {val_f1:^8.4f}")

    # Zapisywanie najlepszego modelu

 Epoch  | Train Loss |  Val Loss  |  Train F1  |   Val F1  
------------------------------------------------------------
   1    |  0.0398  |  0.0398  |  0.4662  |  0.4662 
   2    |  0.0343  |  0.0342  |  0.5786  |  0.5786 
   3    |  0.0312  |  0.0312  |  0.6380  |  0.6380 
   4    |  0.0292  |  0.0292  |  0.6706  |  0.6706 
   5    |  0.0284  |  0.0284  |  0.6660  |  0.6660 
   6    |  0.0264  |  0.0264  |  0.7005  |  0.7005 
   7    |  0.0256  |  0.0256  |  0.7165  |  0.7165 
   8    |  0.0244  |  0.0244  |  0.7366  |  0.7366 
   9    |  0.0237  |  0.0236  |  0.7446  |  0.7446 
  10    |  0.0235  |  0.0235  |  0.7455  |  0.7455 
  11    |  0.0226  |  0.0226  |  0.7671  |  0.7671 
  12    |  0.0220  |  0.0220  |  0.7529  |  0.7529 
  13    |  0.0220  |  0.0220  |  0.7688  |  0.7688 
  14    |  0.0212  |  0.0212  |  0.7733  |  0.7733 
  15    |  0.0209  |  0.0209  |  0.7793  |  0.7793 
  16    |  0.0203  |  0.0202  |  0.7853  |  0.7853 
  17    |  0.0198  |  0.0197  |  0.7964  |  0.7

In [ ]:
test_df = pd.read_parquet("/content/chebi_dataset_test_empty.parquet")

In [ ]:
maccs_fp = MACCSFingerprint(n_jobs=-1)
X_test_maccs = maccs_fp.transform(test_df['SMILES'])
df_maccs_test = pd.DataFrame(X_test_maccs, columns=[f"MACCS_{i}" for i in range(X_test_maccs.shape[1])])

In [ ]:
model_name = "DeepChem/ChemBERTa-77M-MTR"
tokenizer = AutoTokenizer.from_pretrained(model_name)
chemberta = AutoModel.from_pretrained(model_name).to(device)
chemberta.eval()
def get_chemberta_embeddings(smiles_series, batch_size=128):
    """Zamienia listę SMILES na macierz embeddingów z ChemBERTa."""
    embeddings = []
    smiles_list = smiles_series.tolist()

    # Upewniamy się, że model jest na odpowiednim urządzeniu
    chemberta.to(device)
    chemberta.eval()

    with torch.no_grad():
        for i in tqdm(range(0, len(smiles_list), batch_size), desc="🚀 Ekstrakcja ChemBERTa"):
            batch_smiles = smiles_list[i : i + batch_size]

            # Tokenizacja
            inputs = tokenizer(batch_smiles, padding=True, truncation=True,
                               return_tensors="pt", max_length=512).to(device)

            # Forward pass
            outputs = chemberta(**inputs)

            # Wyciągamy [CLS] token (reprezentacja całej cząsteczki)
            cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            embeddings.append(cls_embeddings)

    return np.vstack(embeddings)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/420 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/14.0M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/53 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: DeepChem/ChemBERTa-77M-MTR
Key                             | Status     | 
--------------------------------+------------+-
regression.dense.bias           | UNEXPECTED | 
norm_std                        | UNEXPECTED | 
regression.dense.weight         | UNEXPECTED | 
regression.out_proj.weight      | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
regression.out_proj.bias        | UNEXPECTED | 
norm_mean                       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
print("🧪 Rozpoczynam ekstrakcję embeddingów ChemBERTa...")
X_test_chemberta = get_chemberta_embeddings(test_df['SMILES'])
df_chemberta_test = pd.DataFrame(X_test_chemberta, columns=[f"ChemBERTa_{i}" for i in range(X_test_chemberta.shape[1])])

# KROK 3: Łączenie i synchronizacja kolumn
print("🔗 Łączenie cech i wyrównywanie kolumn...")
X_test_final = pd.concat([df_chemberta_test, df_maccs_test], axis=1)

# Sortowanie kolumn – dodajemy informację, by nie było ciszy
X_test_final = X_test_final[X_train_final.columns]
print(f"✅ Dane testowe przygotowane. Kształt: {X_test_final.shape}")

🧪 Rozpoczynam ekstrakcję embeddingów ChemBERTa...


🚀 Ekstrakcja ChemBERTa: 100%|██████████| 88/88 [12:03<00:00,  8.22s/it]


🔗 Łączenie cech i wyrównywanie kolumn...
✅ Dane testowe przygotowane. Kształt: (11223, 550)


In [ ]:
model.eval()
X_test_tensor = torch.tensor(X_test_final.values, dtype=torch.float32).to(device)

# 2. Generowanie predykcji
print("Generowanie predykcji...")
with torch.no_grad():
    logits = model(X_test_tensor)
    # Nakładamy sigmoid, aby uzyskać prawdopodobieństwa
    probs = torch.sigmoid(logits)
    # Próg 0.5 zamienia na 0 lub 1 (zgodnie z Twoim mappingiem)
    preds_test = (probs > 0.5).int().cpu().numpy()

# 3. Budowanie końcowej ramki danych
class_columns = [f"class_{i}" for i in range(500)]
df_preds = pd.DataFrame(preds_test, columns=class_columns)

# Łączymy mol_id i SMILES z predykcjami
df_submission = pd.concat([test_df[['mol_id', 'SMILES']].reset_index(drop=True), df_preds], axis=1)
df_submission

Generowanie predykcji...


,mol_id,SMILES,class_0,class_1,class_2,class_3,class_4,class_5,class_6,class_7,...,class_490,class_491,class_492,class_493,class_494,class_495,class_496,class_497,class_498,class_499
0,mol_6861,OC[C@H]1O[C@H](O[C@H]2C(O)O[C@H](CO)[C@@H](O)[...,1,1,1,1,1,1,0,1,...,0,0,0,0,0,0,0,0,0,0
1,mol_29793,O.O=C([O-])CC(O)(CC(=O)[O-])C(=O)[O-].[K+].[K+...,1,1,1,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
2,mol_26953,CC(C)=CCC/C(C)=C/CC/C(C)=C/CC/C(C)=C\CC/C(C)=C...,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
3,mol_26053,C=C1CCC(C(C)C)CC1,1,1,1,1,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
4,mol_18653,O=C(O)C1CCCCC1=O,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11218,mol_13480,Oc1cc2cc[nH]c2cc1O,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
11219,mol_42067,CNCC(=O)OCc1cccnc1N(C)C(=O)OC(C)[n+]1cnn(C[C@]...,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
11220,mol_43978,N[C@@H](CCC(=O)[O-])C(O)=Nc1ccc2ccccc2c1,1,1,1,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
11221,mol_16268,COc1cc(-c2[o+]c3cc(O)cc(O)c3cc2O[C@@H]2O[C@H](...,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0


In [ ]:

# 4. Zapis do Parquet
output_filename = '/content/chebi_submission_example.parquet'
df_submission.to_parquet(output_filename, index=False)

print(f"Sukces! Plik '{output_filename}' jest gotowy do wysłania.")
print(f"Liczba wierszy: {len(df_submission)}, Liczba klas: 500")

Sukces! Plik '/content/chebi_submission_example.parquet' jest gotowy do wysłania.
Liczba wierszy: 11223, Liczba klas: 500


In [ ]:
df = pd.read_parquet('/content/data/chebi_submission_example.parquet')